# figure-studio inside Jupyter

This notebook walks through the recommended notebook workflow:

1. `figure_studio.connect(port=8765)` opens (or auto-starts) a long-running server.
2. Build matplotlib figures as usual.
3. `session.add(fig, name="...")` ships each figure to the server.
4. Display the session in a cell to render the editor inline as an iframe.
5. After editing in the browser, `session.get("name")` fetches the live figure back.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import figure_studio

# Auto-starts a local server if nothing is listening on the port.
session = figure_studio.connect(port=8765)
session.url()

## Push a few figures into the session

In [ ]:
rng = np.random.default_rng(0)

fig1, ax = plt.subplots(figsize=(5, 3))
x = np.linspace(0, 4 * np.pi, 200)
ax.plot(x, np.sin(x), label="sin")
ax.plot(x, np.cos(x), label="cos")
ax.set_title("Trig")
ax.legend()
session.add(fig1, name="trig")

In [ ]:
fig2, ax = plt.subplots(figsize=(5, 3))
ax.bar(["Q1", "Q2", "Q3", "Q4"], [3.2, 4.1, 2.9, 3.7])
ax.set_title("Quarterly accuracy")
session.add(fig2, name="bars")

In [ ]:
fig3, axes = plt.subplots(2, 2, figsize=(7, 4.5))
axes[0, 0].plot(x, np.sin(x))
axes[0, 0].set_title("sin")
axes[0, 1].scatter(rng.standard_normal(40), rng.standard_normal(40), s=14, alpha=0.7)
axes[0, 1].set_title("scatter")
axes[1, 0].bar(["a", "b", "c"], [3, 1, 2])
axes[1, 0].set_title("bars")
axes[1, 1].plot(x, np.exp(-x / 5) * np.sin(x))
axes[1, 1].set_title("damped")
fig3.tight_layout()
session.add(fig3, name="grid")

session.list()

## Display the editor inline

Evaluating `session` in a cell renders the editor as an iframe. Click around in it to edit any figure, then run the next cell to fetch the edited result back into Python.

In [ ]:
session

## Round-trip: fetch the edited figure back

After you've made some edits in the iframe above, run this cell. The server pickles its live figure (with all your edits applied) and we get a fresh `Figure` back.

In [ ]:
edited = session.get("trig")
edited

## Export to PDF

Either through the toolbar **Export PDF** button in the iframe, or programmatically:

In [ ]:
session.export_pdf("trig", "/tmp/trig.pdf")
session.export_code("trig", "/tmp/trig.py")  # standalone Python
print("saved trig.pdf + trig.py to /tmp/")

## Extract one subplot into its own figure

Inside the editor, click on any axes and use the **"⤴ Extract as new plot"** button in the inspector. From Python:

In [ ]:
new_name = session.extract_axes("grid", 2)   # axes index 2 = third subplot
print("extracted to:", new_name)
session.list()